In [ ]:
import os
from google.colab import drive
drive.mount('/content/gdrive')
!pwd
os.chdir('gdrive/My Drive/Color_image_denoising/Models/Deep_Learning/paired_data_folder')
!pwd
!ls

## Load data

In [ ]:
###### $$$ Load from already written $$$ #####
import glob
import scipy.io as sio
import re
import numpy as np

SIDDtest_GCP_Nstep4_data = np.load('.//Saved_vector_input/SIDDtest_vectors_Nstep4_GCP.npz')
noisy_data_all_SIDDtest_GCP_Nstep4 = SIDDtest_GCP_Nstep4_data['arr1']
clean_data_all_SIDDtest_GCP_Nstep4 = SIDDtest_GCP_Nstep4_data['arr2']


## Define Network

In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print("Using device:", device)
# ============================================
# 6️⃣ Define neural network
# ============================================
class Net(nn.Module):
    def __init__(self, input_dim=50, output_dim=50):
        super(Net, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, output_dim)
        )

    def forward(self, x):
        noise = self.model(x)
        clean = x - noise              # residual learning
        return clean

## Training/testing

In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ============================================
# 1️⃣ Setup: device selection
# ============================================
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print("Using device:", device)

# ============================================
# 2️⃣ Example data (replace with yours)
# ============================================
trained_dataset_name = 'SIDDval_GCP_Nstep8'
noisy_dataset_training, clean_dataset_training = noisy_data_all_SIDDval_GCP_Nstep8, clean_data_all_SIDDval_GCP_Nstep8
noisy_dataset_testing, clean_dataset_testing = noisy_data_all_SIDDtest_GCP_Nstep4, clean_data_all_SIDDtest_GCP_Nstep4


X_train, X_val, Y_train, Y_val = train_test_split(noisy_dataset_training, clean_dataset_training, test_size=0.1, random_state=42)
X_test, Y_test = noisy_dataset_testing, clean_dataset_testing

# ============================================
# 1️⃣ Setup: device selection
# ============================================
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print("Using device:", device)

# ============================================
# 5️⃣ Convert to tensors (and move to device later)
# ============================================
X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
Y_val_t   = torch.tensor(Y_val, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
Y_test_t  = torch.tensor(Y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, Y_train_t)
val_dataset   = TensorDataset(X_val_t, Y_val_t)
test_dataset  = TensorDataset(X_test_t, Y_test_t)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)


model = Net(input_dim=192, output_dim=192).to(device)  # move model to GPU


# ============================================
# 7️⃣ Define optimizer & loss
# ============================================
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# ============================================
# 8️⃣ Training loop
# ============================================
num_epochs = 61

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    train_loss = 0
    for X_batch, Y_batch in train_loader:
        # Move data to GPU
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, Y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    # --- Validation ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            pred = model(X_batch)
            loss = criterion(pred, Y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    ### Save trained epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = 'Checkpoints/' + trained_dataset_name + '_' + f'epoch_{epoch+1}.pth'
        torch.save(model.state_dict(), checkpoint_path)


# ============================================
# 9️⃣ Test performance
# ============================================
model.eval()
test_loss = 0
with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        pred = model(X_batch)
        loss = criterion(pred, Y_batch)
        test_loss += loss.item() * X_batch.size(0)
test_loss /= len(test_loader.dataset)

print("\n✅ Test Loss:", test_loss)

## Save predictions

In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ============================================
# 1️⃣ Setup: device selection
# ============================================
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print("Using device:", device)

######****** Now we hope to save for SIDD ##*******########
trained_dataset_name = 'SIDDval_GCP_Nstep8'
epoches = 30

model = Net(input_dim=192, output_dim=192).to(device)  # move model to GPU
checkpoint_path = 'Checkpoints/' + trained_dataset_name + '_' + f'epoch_{epoches}.pth'
model.load_state_dict(torch.load(checkpoint_path))
model.eval()
all_preds = []

with torch.no_grad():
    for X_batch, Y_batch in test_loader:  # we don't need Y here
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)
        pred = model(X_batch)
        all_preds.append(pred.cpu().numpy())  # move to CPU, convert to NumPy

# Concatenate all batches together
Y_pred = np.concatenate(all_preds, axis=0)
print("Predicted shape:", Y_pred.shape)

mdic = {"Predict_mean_vectors": Y_pred, "label": "Predict_mean_vectors_SIDDtest"}
sio.savemat("Predict_mean_vectors_SIDDtest_GCP_Nstep4_trained_by_SIDDval_GCP_Nstep8_epoches30.mat", mdic)
print('finished preditictions')




